In [1]:
# ============================================================
# Cell 1 — Load processed factor rasters and review influence
# ============================================================

from pathlib import Path

import numpy as np
import rasterio

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

factor_paths = {
    "DEM": PROCESSED_DIR / "dem_risk.tif",
    "Slope": PROCESSED_DIR / "slope_risk.tif",
    "Rainfall": PROCESSED_DIR / "rainfall_risk.tif",
    "Distance": PROCESSED_DIR / "distance_risk.tif",
}

for name, path in factor_paths.items():
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float32")
        nodata = src.nodata

    if nodata is not None:
        arr[arr == nodata] = np.nan

    valid = arr[~np.isnan(arr)]

    print("\n" + "=" * 50)
    print(name)
    print("Path:", path)
    print("Mean:", np.mean(valid))
    print("Std:", np.std(valid))
    print("Min:", np.min(valid))
    print("Max:", np.max(valid))


DEM
Path: c:\Projects\floodske\floodske\data\processed\dem_risk.tif
Mean: 0.84189314
Std: 0.12585808
Min: 0.0
Max: 1.0

Slope
Path: c:\Projects\floodske\floodske\data\processed\slope_risk.tif
Mean: 0.9592019
Std: 0.06300274
Min: 0.0
Max: 1.0

Rainfall
Path: c:\Projects\floodske\floodske\data\processed\rainfall_risk.tif
Mean: 0.2533508
Std: 0.16869766
Min: 0.0
Max: 1.0

Distance
Path: c:\Projects\floodske\floodske\data\processed\distance_risk.tif
Mean: 0.75911385
Std: 0.20677109
Min: 0.0
Max: 1.0


In [2]:
# ============================================================
# Cell 2 — Rebuild refined susceptibility without distance factor
# ============================================================

loaded_factors = {}

for name, path in factor_paths.items():
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float32")
        profile = src.profile.copy()
        nodata = src.nodata

    if nodata is not None:
        arr[arr == nodata] = np.nan

    loaded_factors[name] = arr

common_mask = np.logical_and.reduce([
    ~np.isnan(loaded_factors["DEM"]),
    ~np.isnan(loaded_factors["Slope"]),
    ~np.isnan(loaded_factors["Rainfall"]),
])

REFINED_WEIGHTS = {
    "DEM": 0.35,
    "Slope": 0.30,
    "Rainfall": 0.35,
}

refined_index = (
    REFINED_WEIGHTS["DEM"] * loaded_factors["DEM"] +
    REFINED_WEIGHTS["Slope"] * loaded_factors["Slope"] +
    REFINED_WEIGHTS["Rainfall"] * loaded_factors["Rainfall"]
)

refined_index[~common_mask] = np.nan

print("Refined susceptibility index created.")
print("Valid pixels:", np.count_nonzero(common_mask))
print("Min:", np.nanmin(refined_index))
print("Max:", np.nanmax(refined_index))
print("Mean:", np.nanmean(refined_index))

Refined susceptibility index created.
Valid pixels: 73533036
Min: 0.31958407
Max: 0.9004562
Mean: 0.6710959


In [3]:
# ============================================================
# Cell 3 — Save refined susceptibility raster
# ============================================================

from pathlib import Path
import rasterio
import numpy as np

output_path = PROCESSED_DIR / "kenya_flood_susceptibility_refined.tif"

profile.update(
    dtype="float32",
    nodata=-9999,
    compress="lzw"
)

output_arr = np.where(
    np.isnan(refined_index),
    -9999,
    refined_index
).astype("float32")

with rasterio.open(output_path, "w", **profile) as dst:
    dst.write(output_arr, 1)

print(output_path)

c:\Projects\floodske\floodske\data\processed\kenya_flood_susceptibility_refined.tif


In [4]:
# ============================================================
# Cell 4 — Classify refined flood susceptibility
# ============================================================

valid_values = refined_index[~np.isnan(refined_index)]

q20, q40, q60, q80 = np.percentile(
    valid_values,
    [20, 40, 60, 80]
)

refined_class = np.zeros(
    refined_index.shape,
    dtype=np.uint8
)

refined_class[~np.isnan(refined_index) & (refined_index <= q20)] = 1
refined_class[~np.isnan(refined_index) & (refined_index > q20) & (refined_index <= q40)] = 2
refined_class[~np.isnan(refined_index) & (refined_index > q40) & (refined_index <= q60)] = 3
refined_class[~np.isnan(refined_index) & (refined_index > q60) & (refined_index <= q80)] = 4
refined_class[~np.isnan(refined_index) & (refined_index > q80)] = 5

unique, counts = np.unique(refined_class, return_counts=True)

for value, count in zip(unique, counts):
    print(value, f"{count:,}")

0 49,913,944
1 14,706,628
2 14,706,608
3 14,706,597
4 14,706,602
5 14,706,601


In [5]:
# ============================================================
# Cell 5 — Save refined classified raster
# ============================================================

refined_class_output = PROCESSED_DIR / "kenya_flood_susceptibility_refined_class.tif"

class_profile = profile.copy()
class_profile.update(
    dtype="uint8",
    nodata=0,
    compress="lzw"
)

with rasterio.open(refined_class_output, "w", **class_profile) as dst:
    dst.write(refined_class, 1)

print(refined_class_output)

c:\Projects\floodske\floodske\data\processed\kenya_flood_susceptibility_refined_class.tif
